# ComplianceGPT — Contract Acceptance Test
This notebook validates **Generator → Pipeline → Verifier** behavior:
- `PRESERVE` == `PARAMS_REQUIRED` + **literal placeholder preservation**
- `NO_EVIDENCE` == **no injected spans**
- Gold-row runs must return `verifier_pass=True` (no hard errors)


## 1) Mount Drive (Colab)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

DRIVE_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"

CCS_REV5 = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
CCS_REV4 = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

# Gold sets
GOLD_REV5 = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev5_gold-set_100q.csv"
GOLD_REV4 = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev4_gold-set_36q.csv"

ANSWERER_PIPELINE_DIR = f"{DRIVE_ROOT}/src/compliancegpt/pipeline"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# Runtime knobs
TOP_K = 10
DOC_FILTER_MODE = "all"
USE_GENERATOR = True           # True = end-to-end; False = pipeline-only extractive fallback
STRICT_CCS_ASSERT = True
MIN_CCS_DOCS = 1000


## 2) Imports + Path Setup
add answerer folder to `sys.path` and imports the pipeline class.

In [3]:
import os, sys
from pathlib import Path

# IMPORTANT: add the repo's /src (package root), not a subfolder.
SRC_DIR = str(Path(DRIVE_ROOT, "src").resolve())
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
print("[sys.path] added:", SRC_DIR)

# Sanity check files exist
for fp in [CCS_REV5, CCS_REV4, GOLD_REV5, GOLD_REV4]:
    print("exists?", fp, "->", Path(fp).exists())

from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline


[sys.path] added: /content/drive/MyDrive/ComplianceGPT_v2/src
exists? /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl -> True
exists? /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl -> True
exists? /content/drive/MyDrive/ComplianceGPT_v2/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev5_gold-set_100q.csv -> True
exists? /content/drive/MyDrive/ComplianceGPT_v2/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev4_gold-set_36q.csv -> True
[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib


## 3) Load Gold Sets + Quick Sanity
We verify `PRESERVE` exists and print counts.

In [4]:
import pandas as pd

def read_csv_robust(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"[OK] read with encoding={enc}, rows={len(df)}, cols={len(df.columns)}")
            return df
        except UnicodeDecodeError as e:
            print(f"[FAIL] encoding={enc}: {e}")
    # last resort: force-read and replace bad characters
    df = pd.read_csv(path, encoding="utf-8", encoding_errors="replace")
    print(f"[WARN] forced read with replacement, rows={len(df)}")
    return df

gold5 = read_csv_robust(GOLD_REV5)
gold4 = read_csv_robust(GOLD_REV4)


[FAIL] encoding=utf-8: 'utf-8' codec can't decode byte 0xd1 in position 529: invalid continuation byte
[FAIL] encoding=utf-8-sig: 'utf-8' codec can't decode byte 0xd1 in position 47789: invalid continuation byte
[OK] read with encoding=cp1252, rows=100, cols=12
[OK] read with encoding=utf-8, rows=36, cols=12


## 4) Initialize Pipelines (Rev5 + Rev4)
This loads the model once per pipeline and initializes retriever/generator/verifier.

In [5]:
!pip -q install -U "bitsandbytes>=0.46.1" sentence-transformers faiss-cpu rank-bm25 accelerate

import torch
if torch.cuda.is_available():
    !pip -q install bitsandbytes
print("cuda:", torch.cuda.is_available(), "torch:", torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 106.4 MB/s eta 0:00:00
cuda: True torch: 2.9.0+cu126


In [6]:

pipe5 = ComplianceGPTPipeline(
    framework_version="rev5",
    model_id=MODEL_ID,
    use_qur=True,
    doc_filter_mode=DOC_FILTER_MODE,
    ccs_path=CCS_REV5,
    strict_ccs_assert=STRICT_CCS_ASSERT,
    min_ccs_docs=MIN_CCS_DOCS,
)

pipe4 = ComplianceGPTPipeline(
    framework_version="rev4",
    model_id=MODEL_ID,
    use_qur=True,
    doc_filter_mode=DOC_FILTER_MODE,
    ccs_path=CCS_REV4,
    strict_ccs_assert=STRICT_CCS_ASSERT,
    min_ccs_docs=MIN_CCS_DOCS,
)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 5) Acceptance Test A — Rev5 PRESERVE (id=1)
Expected:
- `status=PARAMS_REQUIRED`
- placeholders preserved
- `verifier_pass=True`

In [7]:

row = gold5[gold5["id"] == 1].iloc[0].to_dict()
out = pipe5.answer(row["question"], gold_row=row, top_k=TOP_K, use_generator=USE_GENERATOR)

c = out["contract"]
print("status:", c.get("status"))
print("odp_required_list:", c.get("odp_required_list"))
print("evidence_spans_len:", len(c.get("evidence_spans", [])))
print("answer_text_preview:", repr((c.get("answer_text","") or "")[:120]))
print("verifier_pass:", c.get("verifier_pass"))
print("verifier_errors:", c.get("verifier_errors", []))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


status: PARAMS_REQUIRED
odp_required_list: ['ac-02_odp.01', 'ac-02_odp.02', 'ac-02_odp.03', 'ac-02_odp.04', 'ac-02_odp.05', 'ac-02_odp.06', 'ac-02_odp.07', 'ac-02_odp.08', 'ac-02_odp.09', 'ac-02_odp.10']
evidence_spans_len: 2
answer_text_preview: 'a. Define and document the types of accounts allowed and specifically prohibited for use within the system; b. Assign ac'
verifier_pass: False
verifier_errors: []


## 6) Acceptance Test B — Rev4 PRESERVE (id=21)
Expected:
- `status=PARAMS_REQUIRED`
- placeholders preserved
- `verifier_pass=True`

In [8]:

row = gold4[gold4["id"] == 21].iloc[0].to_dict()
out = pipe4.answer(row["question"], gold_row=row, top_k=TOP_K, use_generator=USE_GENERATOR)

c = out["contract"]
print("status:", c.get("status"))
print("odp_required_list:", c.get("odp_required_list"))
print("evidence_spans_len:", len(c.get("evidence_spans", [])))
print("answer_text_preview:", repr((c.get("answer_text","") or "")[:120]))
print("verifier_pass:", c.get("verifier_pass"))
print("verifier_errors:", c.get("verifier_errors", []))


status: PARAMS_REQUIRED
odp_required_list: ['pe-16_prm_1', 'pe-3_prm_1', 'pe-3_prm_2', 'pe-3_prm_4', 'pe-3_prm_5', 'pe-3_prm_6', 'pe-3_prm_7', 'pe-3_prm_8', 'pe-3_prm_9']
evidence_spans_len: 2
answer_text_preview: 'The organization: a. Enforces physical access authorizations at {{ insert: param, pe-3_prm_1 }} by; 1. Verifying individ'
verifier_pass: False
verifier_errors: []


## 7) Acceptance Test C — NO_EVIDENCE Sanity
Expected:
- `status=NO_EVIDENCE`
- `evidence_spans_len=0`
- `answer_text` is `''` or `'NO_EVIDENCE'`

In [9]:

out = pipe5.answer("zzzxqv not a real control question", gold_row=None, top_k=TOP_K, use_generator=USE_GENERATOR)
c = out["contract"]

print("status:", c.get("status"))
print("selected_source_ids:", c.get("selected_source_ids", []))
print("odp_required_list:", c.get("odp_required_list", []))
print("evidence_spans_len:", len(c.get("evidence_spans", [])))
print("answer_text_preview:", repr((c.get("answer_text","") or "")[:120]))


status: NO_EVIDENCE
selected_source_ids: []
odp_required_list: []
evidence_spans_len: 0
answer_text_preview: ''


print

In [10]:
import json, os, datetime

def _to_jsonable(x):

    try:
        json.dumps(x)
        return x
    except Exception:
        return str(x)

def snapshot_row(pipe, gold_df, row_id, name="rev5", top_k=None, use_generator=None, save_path=None):
    row = gold_df[gold_df["id"] == row_id].iloc[0].to_dict()
    out = pipe.answer(
        row["question"],
        gold_row=row,
        top_k=TOP_K if top_k is None else top_k,
        use_generator=USE_GENERATOR if use_generator is None else use_generator,
    )
    c = out.get("contract", {}) or {}
    dbg = c.get("debug", {}) or {}

    print("\n===== SNAPSHOT", name, "id", row_id, "=====")
    print("id:", row.get("id"), "control:", row.get("control_id"))
    print("policy_raw:", row.get("resolution_policy"),
          "policy_used:", dbg.get("resolution_policy"),
          "doc_filter_mode_used:", dbg.get("doc_filter_mode"),
          "keep_kinds:", dbg.get("keep_kinds"))

    print("status:", c.get("status"))
    print("odp_required_list:", c.get("odp_required_list"))
    print("verifier_pass:", c.get("verifier_pass"))
    print("verifier_errors:", c.get("verifier_errors"))

    print("verification(raw):", c.get("verification"))

    print("\n[debug] retrieved_control_order (first 10):", (dbg.get("retrieved_control_order") or [])[:10])
    print("[debug] retrieved_doc_ids (first 20):", (dbg.get("retrieved_doc_ids") or [])[:20])
    print("[debug] docs_for_gen_ids (first 20):", (dbg.get("docs_for_gen_ids") or [])[:20])
    print("[debug] selected_source_ids:", (dbg.get("selected_source_ids") or [])[:20])


    snap = {
        "meta": {
            "name": name,
            "row_id": row_id,
            "ts": datetime.datetime.now().isoformat(timespec="seconds"),
        },
        "row": {k: _to_jsonable(v) for k, v in row.items()},
        "contract": _to_jsonable(c),
        "debug": _to_jsonable(dbg),
    }

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(snap, f, ensure_ascii=False, indent=2)
        print("\nSaved snapshot to:", save_path)

    return out, snap


## 8) Small Batch Sanity Run (10 rows each)
This is not the full evaluation; it’s a quick regression check.
We report pass rates overall and by `resolution_policy`.

In [11]:

import random
import pandas as pd

def run_batch(pipe, df, n=10, seed=7):
    import random, pandas as pd
    rng = random.Random(seed)
    idxs = list(df.index)
    rng.shuffle(idxs)
    idxs = idxs[:min(n, len(idxs))]

    rows = []
    for i in idxs:
        row = df.loc[i].to_dict()
        out = pipe.answer(row["question"], gold_row=row, top_k=TOP_K, use_generator=USE_GENERATOR)
        c = out["contract"]
        dbg = c.get("debug", {}) or {}

        rows.append({
            "id": row.get("id"),
            "control_id": row.get("control_id"),

            # what CSV says (legacy)
            "resolution_policy_raw": str(row.get("resolution_policy","")).upper().strip(),

            # what pipeline actually used
            "policy_used": str(dbg.get("resolution_policy","")).upper().strip(),
            "doc_filter_used": str(dbg.get("doc_filter_mode","")),

            "status": str(c.get("status","")).upper().strip(),
            "verifier_pass": bool(c.get("verifier_pass", False)),
            "n_spans": len(c.get("evidence_spans", []) or []),
            "errors": ";".join((c.get("verifier_errors", []) or [])[:5]),
        })
    return pd.DataFrame(rows)


df5 = run_batch(pipe5, gold5, n=10, seed=5)
df4 = run_batch(pipe4, gold4, n=10, seed=4)

print("=== rev5 sample ===")
display(df5)
print("\n=== rev4 sample ===")
display(df4)

def summarize(df, col="policy_used"):
    overall = float(df["verifier_pass"].mean()) if len(df) else 0.0
    by = df.groupby(col)["verifier_pass"].mean().sort_index() if len(df) else None
    return overall, by

overall5, by5 = summarize(df5, "policy_used")
overall4, by4 = summarize(df4, "policy_used")

print("rev5 overall:", overall5); print(by5)
print("rev4 overall:", overall4); print(by4)

print("\nrev5 by raw CSV policy:")
print(summarize(df5, "resolution_policy_raw")[1])

print("\nrev4 by raw CSV policy:")
print(summarize(df4, "resolution_policy_raw")[1])


fails5 = df5[df5["verifier_pass"] == False]
fails4 = df4[df4["verifier_pass"] == False]
if len(fails5):
    print("\nrev5 failures:")
    display(fails5)
if len(fails4):
    print("\nrev4 failures:")
    display(fails4)


=== rev5 sample ===


,id,control_id,resolution_policy_raw,policy_used,doc_filter_used,status,verifier_pass,n_spans,errors
0,92,SI-3,ASK,ASK,all,PARAMS_REQUIRED,False,6,
1,3,AC-6,NAN,ASK,all,OK,False,2,
2,31,IA-2,NAN,ASK,all,OK,False,2,
3,34,IA-6,NAN,ASK,all,OK,False,2,
4,59,PL-8,FILL_FROM_PROFILE,FILL_FROM_PROFILE,all,OK,False,2,
5,69,PS-5,PRESERVE,PRESERVE,all,OK,False,4,
6,86,SC-7,FILL_FROM_PROFILE,FILL_FROM_PROFILE,all,PARAMS_REQUIRED,False,2,
7,8,AT-4,FILL_FROM_PROFILE,FILL_FROM_PROFILE,all,PARAMS_REQUIRED,False,2,
8,75,PT-6,NAN,ASK,all,OK,False,3,
9,30,CP-9,PRESERVE,PRESERVE,all,PARAMS_REQUIRED,False,3,



=== rev4 sample ===


,id,control_id,resolution_policy_raw,policy_used,doc_filter_used,status,verifier_pass,n_spans,errors
0,25,PM-1,NAN,ASK,all,PARAMS_REQUIRED,False,3,
1,24,PL-4,NAN,ASK,all,PARAMS_REQUIRED,False,2,
2,23,PL-2,NAN,ASK,all,OK,False,2,
3,14,IA-5,NAN,ASK,all,PARAMS_REQUIRED,False,2,
4,34,SC-8,FILL_FROM_PROFILE,FILL_FROM_PROFILE,all,OK,False,5,
5,21,PE-3,PRESERVE,PRESERVE,all,PARAMS_REQUIRED,False,2,
6,36,SI-4,ASK,ASK,all,PARAMS_REQUIRED,False,2,
7,27,PS-2,NAN,ASK,all,PARAMS_REQUIRED,False,2,
8,29,RA-2,NAN,ASK,all,OK,False,3,
9,15,IR-4,NAN,ASK,all,OK,False,2,


rev5 overall: 0.0
policy_used
ASK                  0.0
FILL_FROM_PROFILE    0.0
PRESERVE             0.0
Name: verifier_pass, dtype: float64
rev4 overall: 0.0
policy_used
ASK                  0.0
FILL_FROM_PROFILE    0.0
PRESERVE             0.0
Name: verifier_pass, dtype: float64

rev5 by raw CSV policy:
resolution_policy_raw
ASK                  0.0
FILL_FROM_PROFILE    0.0
NAN                  0.0
PRESERVE             0.0
Name: verifier_pass, dtype: float64

rev4 by raw CSV policy:
resolution_policy_raw
ASK                  0.0
FILL_FROM_PROFILE    0.0
NAN                  0.0
PRESERVE             0.0
Name: verifier_pass, dtype: float64

rev5 failures:


,id,control_id,resolution_policy_raw,policy_used,doc_filter_used,status,verifier_pass,n_spans,errors
0,92,SI-3,ASK,ASK,all,PARAMS_REQUIRED,False,6,
1,3,AC-6,NAN,ASK,all,OK,False,2,
2,31,IA-2,NAN,ASK,all,OK,False,2,
3,34,IA-6,NAN,ASK,all,OK,False,2,
4,59,PL-8,FILL_FROM_PROFILE,FILL_FROM_PROFILE,all,OK,False,2,
5,69,PS-5,PRESERVE,PRESERVE,all,OK,False,4,
6,86,SC-7,FILL_FROM_PROFILE,FILL_FROM_PROFILE,all,PARAMS_REQUIRED,False,2,
7,8,AT-4,FILL_FROM_PROFILE,FILL_FROM_PROFILE,all,PARAMS_REQUIRED,False,2,
8,75,PT-6,NAN,ASK,all,OK,False,3,
9,30,CP-9,PRESERVE,PRESERVE,all,PARAMS_REQUIRED,False,3,



rev4 failures:


,id,control_id,resolution_policy_raw,policy_used,doc_filter_used,status,verifier_pass,n_spans,errors
0,25,PM-1,NAN,ASK,all,PARAMS_REQUIRED,False,3,
1,24,PL-4,NAN,ASK,all,PARAMS_REQUIRED,False,2,
2,23,PL-2,NAN,ASK,all,OK,False,2,
3,14,IA-5,NAN,ASK,all,PARAMS_REQUIRED,False,2,
4,34,SC-8,FILL_FROM_PROFILE,FILL_FROM_PROFILE,all,OK,False,5,
5,21,PE-3,PRESERVE,PRESERVE,all,PARAMS_REQUIRED,False,2,
6,36,SI-4,ASK,ASK,all,PARAMS_REQUIRED,False,2,
7,27,PS-2,NAN,ASK,all,PARAMS_REQUIRED,False,2,
8,29,RA-2,NAN,ASK,all,OK,False,3,
9,15,IR-4,NAN,ASK,all,OK,False,2,


In [12]:
def drill(pipe, gold_df, row_id):
    row = gold_df[gold_df["id"] == row_id].iloc[0].to_dict()

    out = pipe.answer(
        row["question"],
        gold_row=row,
        top_k=TOP_K,
        use_generator=USE_GENERATOR
    )

    c = out["contract"]
    dbg = c.get("debug", {}) or {}

    print("id:", row.get("id"),
          "control:", row.get("control_id"),
          "policy_raw:", row.get("resolution_policy"),
          "policy_used:", dbg.get("resolution_policy"),
          "doc_filter_used:", dbg.get("doc_filter_mode"))

    print("status:", c.get("status"))
    print("odp_required_list:", c.get("odp_required_list"))
    print("verifier_pass:", c.get("verifier_pass"))
    print("verifier_errors:", c.get("verifier_errors", []))

    print("\n[debug] retrieved_control_order (first 10):",
          (dbg.get("retrieved_control_order") or [])[:10])
    print("[debug] retrieved_doc_ids (first 15):",
          (dbg.get("retrieved_doc_ids") or [])[:15])
    print("[debug] docs_for_gen_ids (first 15):",
          (dbg.get("docs_for_gen_ids") or [])[:15])
    print("[debug] selected_source_ids:",
          (dbg.get("selected_source_ids") or [])[:20])

    return out


In [13]:
out_92 = drill(pipe5, gold5, 92)
c = out_92["contract"]
dbg = c.get("debug", {}) or {}

print("doc_filter_used:", dbg.get("doc_filter_mode"))
print("added_param_doc_ids (first 20):", (dbg.get("added_param_doc_ids") or [])[:20])

docs_for_gen = dbg.get("docs_for_gen_ids") or []
print("docs_for_gen has odp/prm?:",
      [x for x in docs_for_gen if ("_odp" in str(x) or "_prm" in str(x))][:30])

print("verification.ok:", (c.get("verification") or {}).get("ok"))
print("verifier_errors (first 10):", (c.get("verifier_errors") or [])[:10])


id: 92 control: SI-3 policy_raw: ASK policy_used: ASK doc_filter_used: all
status: PARAMS_REQUIRED
odp_required_list: ['si-03.06_odp', 'si-03_odp.01', 'si-03_odp.02', 'si-03_odp.03', 'si-03_odp.04', 'si-03_odp.06']
verifier_pass: False
verifier_errors: []

[debug] retrieved_control_order (first 10): ['SI-3', 'PL-8', 'AT-2', 'MA-4', 'SI-4', 'SI-2', 'IR-4']
[debug] retrieved_doc_ids (first 15): ['si-3_smt', 'si-3_gdn', 'si-3_smt.a', 'si-3.6_smt', 'si-3.6_smt.a', 'si-3.6_smt.b', 'si-3.4_gdn', 'si-3.4_smt', 'pl-8.2_gdn', 'pl-8.2_smt']
[debug] docs_for_gen_ids (first 15): ['si-3_smt', 'si-3_gdn', 'si-3_smt.a', 'si-3.6_smt', 'si-3.6_smt.a', 'si-3.6_smt.b', 'si-3.4_gdn', 'si-3.4_smt', 'pl-8.2_gdn', 'pl-8.2_smt']
[debug] selected_source_ids: ['si-3_smt', 'si-3_gdn', 'si-3_smt.a', 'si-3.4_smt', 'si-3.6_smt']
doc_filter_used: all
added_param_doc_ids (first 20): []
docs_for_gen has odp/prm?: []
verification.ok: False
verifier_errors (first 10): []
